# Silver → Gold (model training layer)
Drops leakage and pipeline-artifact columns from the silver dataset
and saves a clean, training-ready CSV to `data/gold/gold_model.csv`.

Run this notebook whenever the silver layer changes, before retraining any model.

**Columns removed vs silver:**
| Column | Reason |
|---|---|
| `normalized_date` | Free-text date string — not useful without parsing |
| `is_duplicate` | Data-pipeline flag, not a real credit-risk signal |
| `final_decision` | Post-hoc label — decided after default is observed (leakage) |
| `requires_human_review` | Post-hoc pipeline flag (leakage) |

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from config import SILVER_DATASET, GOLD_MODEL, GOLD_DATA_DIR

In [2]:
df = pd.read_csv(SILVER_DATASET)
print(f"Silver  rows={len(df):,}  cols={df.shape[1]}")

Silver  rows=24,974  cols=39


In [3]:
DROP = ['normalized_date', 'is_duplicate', 'final_decision', 'requires_human_review']
df_model = df.drop(columns=[c for c in DROP if c in df.columns])

print(f"Gold model  rows={len(df_model):,}  cols={df_model.shape[1]}")
print(f"Dropped : {DROP}")
print(f"\nTarget distribution:")
print(df_model['default_12m'].value_counts().rename({0:'Adimplente', 1:'Inadimplente'}))
print(f"Default rate: {df_model['default_12m'].mean()*100:.1f}%")
df_model.head(2)

Gold model  rows=24,974  cols=35
Dropped : ['normalized_date', 'is_duplicate', 'final_decision', 'requires_human_review']

Target distribution:
default_12m
Adimplente      20822
Inadimplente     4152
Name: count, dtype: int64
Default rate: 16.6%


,source_system,ocr_engine,doc_type,document_image_quality,ocr_confidence,ocr_error_count,text_language,normalized_amount,normalization_method,match_score,...,customer_segment,industry_sector,credit_requested_value,income_declared,tenure_months,collateral_type,ltv,pd_model_score,default_12m,env_risk_level
0,OCR_PDF,AZURE_OCR,EXTRATO,0.821,0.562,4,PT,10348.78,RULES_V1,0.720,...,PJ_ME,PF,10898.86,53421.80,210,IMOVEL,0.67,0.297,0,BAIXO
1,EMAIL_ATTACH,GOOGLE_VISION,EXTRATO,0.622,0.830,1,PT,21423.14,RULES_V2,0.659,...,PF,INDUSTRIA,26290.89,28138.01,152,VEICULO,0.67,0.223,1,BAIXO


In [4]:
GOLD_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_model.to_csv(GOLD_MODEL, index=False)
print(f"Saved → {GOLD_MODEL}")
print(f"Columns ({df_model.shape[1]}):")
print(df_model.columns.tolist())

Saved → /home/paolot/Workspace/BB_data_analysis/data/gold/gold_model.csv
Columns (35):
['source_system', 'ocr_engine', 'doc_type', 'document_image_quality', 'ocr_confidence', 'ocr_error_count', 'text_language', 'normalized_amount', 'normalization_method', 'match_score', 'join_status', 'uf', 'regiao', 'data_quality_score', 'rule_violations', 'pii_detected', 'compliance_status', 'bioma', 'precip_mm_30d', 'drought_spi', 'flood_risk_idx', 'ndvi', 'fire_hotspots_30d', 'deforestation_km2_12m', 'climate_alert_level', 'customer_segment', 'industry_sector', 'credit_requested_value', 'income_declared', 'tenure_months', 'collateral_type', 'ltv', 'pd_model_score', 'default_12m', 'env_risk_level']
